In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import cooler
import cooltools
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm
from scipy.stats import pearsonr

In [ ]:
projdir = '/u/project/cluo/terencew/claude/igvf/pgp/ipsc_le_se_rerun'
pubdir = '/u/project/cluo/terencew/igvf/2023_YR2/snm3C/pseudobulk_hic'
outdir = f'{projdir}/csv'
samples = ['IPS_main', 'IPS_alt']
res = 100000
autosome = [f'chr{x}' for x in range(1, 23)]

### cis eigenvectors, as in 09_ipsc_subtypes/08_get_comps_ips_cluster (early-iPSC mcool)

In [ ]:
gc_cov = pd.read_csv('/u/project/cluo/terencew/reference/hg38/bed/hg38_100kb_gc.bed', sep='\t', header=0, index_col=0)
gc_cov['chrom'] = [f'chr{x}' for x in gc_cov['chrom']]
gc_cov.shape

In [ ]:
gc_cov.head()

In [ ]:
def process_eigs(eigs, cutoff=0):
    for e in ['E1', 'E2', 'E3']:
        eigs[f'{e}_comp'] = np.where(eigs[e] < cutoff, 'B', 'A')
    return eigs

def get_comps(s):
    clr = cooler.Cooler(f'{projdir}/cooler/ips_cluster/{s}.mcool::/resolutions/{res}')
    cis_eigs = cooltools.eigs_cis(clr, gc_cov, clr_weight_name='weight', n_eigs=3)
    eig_tracks = cis_eigs[0]
    eig_tracks['sample'] = s
    eigs = cis_eigs[1]
    auto_eigs = eigs[eigs['chrom'].isin(autosome)].dropna()
    auto_eigs = process_eigs(auto_eigs)
    auto_eigs['sample'] = s
    return (eig_tracks, auto_eigs)

In [ ]:
with ProcessPoolExecutor(max_workers=2) as executor:
    results = list(tqdm(executor.map(get_comps, samples), total=len(samples)))
tracks = pd.concat([x[0] for x in results]).reset_index(drop=True)
comps = pd.concat([x[1] for x in results]).reset_index(drop=True)
tracks.shape, comps.shape

In [ ]:
comps.head()

In [ ]:
comps.groupby('sample')['E1_comp'].value_counts()

In [ ]:
tracks.to_csv(f'{outdir}/ips_cluster_compartments_tracks_100kb.csv', sep='\t')
comps.to_csv(f'{outdir}/ips_cluster_compartments_comps_100kb.csv', sep='\t')

### shared bins with 4DN RUES2 iPSC, as in 09_ipsc_subtypes/10_compare_comps_to_4dn

In [ ]:
comps.index = [f'{x}_{y}' for x, y in zip(comps['chrom'], comps['start'])]
alt_comps = comps[comps['sample'] == 'IPS_alt']
main_comps = comps[comps['sample'] == 'IPS_main']
ref_comps = pd.read_csv('/u/project/cluo/terencew/public_datasets/4dn/compartments/cooltools/ipsc_rues2.100kb.csv', sep='\t')
ref_comps.index = [f'{x}_{y}' for x, y in zip(ref_comps['chrom'], ref_comps['start'])]
alt_comps.shape, main_comps.shape, ref_comps.shape

In [ ]:
final_comps = alt_comps.index.intersection(main_comps.index).intersection(ref_comps.index)
tmp = alt_comps.loc[final_comps].copy()
tmp['chr_num'] = [int(x.split('chr')[1]) for x in tmp['chrom']]
final_comps = tmp.sort_values(by=['chr_num', 'start']).index
eigs_df = pd.DataFrame({'IPS_alt': alt_comps.loc[final_comps, 'E1'], 'IPS_main': main_comps.loc[final_comps, 'E1'], '4DN': ref_comps.loc[final_comps, 'E1']})
comps_df = pd.DataFrame({'IPS_alt': alt_comps.loc[final_comps, 'E1_comp'], 'IPS_main': main_comps.loc[final_comps, 'E1_comp'], '4DN': ref_comps.loc[final_comps, 'E1_comp']})
eigs_df.shape

In [ ]:
eigs_df.head()

### E1 correlation with 4DN (ED10L)

In [ ]:
to_plot = pd.DataFrame({'pearson_r': [pearsonr(eigs_df[s], eigs_df['4DN'])[0] for s in ['IPS_alt', 'IPS_main']],
                        'ips_subtype': ['IPS_alt', 'IPS_main']})
to_plot

In [ ]:
to_plot.to_csv(f'{outdir}/4dn_mature_ips_eigenvector_corr.csv', sep='\t')

### split by 4DN compartment, and compartment concordance

In [ ]:
rows = []
for c in ['A', 'B']:
    sub = eigs_df[comps_df['4DN'] == c]
    for s in ['IPS_alt', 'IPS_main']:
        rows.append([c, s, pearsonr(sub[s], sub['4DN'])[0], (comps_df.loc[sub.index, s] == c).mean()])
by_comp = pd.DataFrame(rows, columns=['comp', 'variable', 'value', 'concordance'])
by_comp

In [ ]:
(comps_df['IPS_alt'] == comps_df['4DN']).mean(), (comps_df['IPS_main'] == comps_df['4DN']).mean()

In [ ]:
by_comp.to_csv(f'{outdir}/4dn_mature_ips_eigenvector_corr_by_comp.csv', sep='\t')

### compare with published (all-nuclei) eigenvectors

In [ ]:
pub = pd.read_csv(f'{pubdir}/csv/compartments/cooltools/ips_cluster/merged.comps.100kb.csv', sep='\t', index_col=0)
pub.index = [f'{x}_{y}' for x, y in zip(pub['chrom'], pub['start'])]
for s in samples:
    a = comps[comps['sample'] == s]
    b = pub[pub['sample'] == s]
    idx = a.index.intersection(b.index)
    print(s, len(idx), pearsonr(a.loc[idx, 'E1'], b.loc[idx, 'E1'])[0], (a.loc[idx, 'E1_comp'] == b.loc[idx, 'E1_comp']).mean())

In [ ]:
!date